# CS201L: Artificial Intelligence Laboratory
## Lab 12: Clustering - K-Means and K-Medoid
**Indian Institute of Technology Dharwad**

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from collections import Counter

## 1. Data Loading and Exploration

In [ ]:
# (a) Load the dataset
tsne_df = pd.read_csv('tsne_2d_features_6_languages.csv')
print('Shape:', tsne_df.shape)
print(tsne_df.head())

In [ ]:
# (b) Separate features and labels
features = tsne_df[['tsne_1', 'tsne_2']].values
true_labels = tsne_df['language'].values

print('Features shape:', features.shape)
print('Unique languages:', np.unique(true_labels))
print('Label distribution:')
print(pd.Series(true_labels).value_counts())

In [ ]:
# (c) Scatter plot of dataset with language labels
languages = np.unique(true_labels)
colors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']

plt.figure(figsize=(9, 7))
for lang, color in zip(languages, colors):
    mask = true_labels == lang
    plt.scatter(features[mask, 0], features[mask, 1],
                label=lang, color=color, alpha=0.5, s=15)

plt.title('Scatter plot of 2D t-SNE features of six language audio embeddings')
plt.xlabel('tsne_1')
plt.ylabel('tsne_2')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_languages.png', dpi=150)
plt.show()

## Helper Function: Purity Score

In [ ]:
def purity_score(true_labels, cluster_labels):
    """
    Compute purity score for clustering.
    Purity = (1/N) * sum_k( max_j( |cluster_k ∩ class_j| ) )
    """
    N = len(true_labels)
    total_correct = 0
    for cluster_id in np.unique(cluster_labels):
        # Get true labels for all points in this cluster
        cluster_mask = cluster_labels == cluster_id
        true_in_cluster = true_labels[cluster_mask]
        # Find the most frequent true label in this cluster
        most_common_count = Counter(true_in_cluster).most_common(1)[0][1]
        total_correct += most_common_count
    return total_correct / N

## 2. K-Means Clustering

In [ ]:
K_values = list(range(2, 11))

kmeans_results = {
    'K': [],
    'Inertia': [],
    'Purity': [],
    'NMI': [],
    'Silhouette': []
}

kmeans_models = {}

for k in K_values:
    # (a) Apply K-Means clustering
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(features)
    
    # (b) Assign cluster labels
    labels = km.labels_
    kmeans_models[k] = km
    
    # (d) Distortion / Inertia
    inertia = km.inertia_
    
    # (e) Purity Score
    purity = purity_score(true_labels, labels)
    
    # (f) NMI
    nmi = normalized_mutual_info_score(true_labels, labels)
    
    # (g) Silhouette Score
    sil = silhouette_score(features, labels)
    
    kmeans_results['K'].append(k)
    kmeans_results['Inertia'].append(round(inertia, 2))
    kmeans_results['Purity'].append(round(purity, 4))
    kmeans_results['NMI'].append(round(nmi, 4))
    kmeans_results['Silhouette'].append(round(sil, 4))
    
    print(f'K={k}: Inertia={inertia:.2f}, Purity={purity:.4f}, NMI={nmi:.4f}, Silhouette={sil:.4f}')

In [ ]:
# (c) Visualize clustered data for each K (K-Means)
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for idx, k in enumerate(K_values):
    km = kmeans_models[k]
    labels = km.labels_
    centers = km.cluster_centers_
    ax = axes[idx]
    scatter = ax.scatter(features[:, 0], features[:, 1],
                         c=labels, cmap='tab10', alpha=0.5, s=10)
    ax.scatter(centers[:, 0], centers[:, 1],
               c='black', marker='X', s=120, label='Centroids', zorder=5)
    ax.set_title(f'K-Means (K={k})')
    ax.set_xlabel('tsne_1')
    ax.set_ylabel('tsne_2')
    ax.legend(fontsize=7)

plt.suptitle('K-Means Clustering Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# (h) Performance metrics table - K-Means
kmeans_df = pd.DataFrame(kmeans_results)
print('=== K-Means Performance Metrics ===')
print(kmeans_df.to_string(index=False))

In [ ]:
# (i) Elbow plot - K-Means
plt.figure(figsize=(8, 5))
plt.plot(kmeans_results['K'], kmeans_results['Inertia'], 'bo-', markersize=8, linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Distortion (Inertia)')
plt.title('K-Means: Elbow Method - K vs Distortion')
plt.xticks(K_values)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('kmeans_elbow.png', dpi=150)
plt.show()
print('\nElbow method: Examine the plot above. The heuristic optimal K is where the inertia curve bends (elbow).')
print('Based on 6 languages in the dataset, the optimal K is likely 6.')

## 3. K-Medoid Clustering

In [ ]:
kmedoid_results = {
    'K': [],
    'Inertia': [],
    'Purity': [],
    'NMI': [],
    'Silhouette': []
}

kmedoid_models = {}

for k in K_values:
    # (a) Apply K-Medoid clustering
    kmed = KMedoids(n_clusters=k, random_state=42, method='pam')
    kmed.fit(features)
    
    # (b) Assign cluster labels
    labels = kmed.labels_
    kmedoid_models[k] = kmed
    
    # (d) Distortion / Inertia
    inertia = kmed.inertia_
    
    # (e) Purity Score
    purity = purity_score(true_labels, labels)
    
    # (f) NMI
    nmi = normalized_mutual_info_score(true_labels, labels)
    
    # (g) Silhouette Score
    sil = silhouette_score(features, labels)
    
    kmedoid_results['K'].append(k)
    kmedoid_results['Inertia'].append(round(inertia, 2))
    kmedoid_results['Purity'].append(round(purity, 4))
    kmedoid_results['NMI'].append(round(nmi, 4))
    kmedoid_results['Silhouette'].append(round(sil, 4))
    
    print(f'K={k}: Inertia={inertia:.2f}, Purity={purity:.4f}, NMI={nmi:.4f}, Silhouette={sil:.4f}')

In [ ]:
# (c) Visualize clustered data for each K (K-Medoid)
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for idx, k in enumerate(K_values):
    kmed = kmedoid_models[k]
    labels = kmed.labels_
    centers = kmed.cluster_centers_
    ax = axes[idx]
    ax.scatter(features[:, 0], features[:, 1],
               c=labels, cmap='tab10', alpha=0.5, s=10)
    ax.scatter(centers[:, 0], centers[:, 1],
               c='black', marker='X', s=120, label='Medoids', zorder=5)
    ax.set_title(f'K-Medoid (K={k})')
    ax.set_xlabel('tsne_1')
    ax.set_ylabel('tsne_2')
    ax.legend(fontsize=7)

plt.suptitle('K-Medoid Clustering Results', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('kmedoid_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# (h) Performance metrics table - K-Medoid
kmedoid_df = pd.DataFrame(kmedoid_results)
print('=== K-Medoid Performance Metrics ===')
print(kmedoid_df.to_string(index=False))

In [ ]:
# (i) Elbow plot - K-Medoid
plt.figure(figsize=(8, 5))
plt.plot(kmedoid_results['K'], kmedoid_results['Inertia'], 'rs-', markersize=8, linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Distortion (Inertia)')
plt.title('K-Medoid: Elbow Method - K vs Distortion')
plt.xticks(K_values)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('kmedoid_elbow.png', dpi=150)
plt.show()
print('\nElbow method: Examine the plot above. The heuristic optimal K is where the inertia curve bends (elbow).')
print('Based on 6 languages in the dataset, the optimal K is likely 6.')

## 4. Comparison: K-Means vs K-Medoid

In [ ]:
# Side-by-side comparison at K=6 (true number of classes)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, title in zip(axes,
                             [kmeans_models[6], kmedoid_models[6]],
                             ['K-Means (K=6)', 'K-Medoid (K=6)']):
    labels = model.labels_
    centers = model.cluster_centers_
    ax.scatter(features[:, 0], features[:, 1],
               c=labels, cmap='tab10', alpha=0.5, s=12)
    ax.scatter(centers[:, 0], centers[:, 1],
               c='black', marker='X', s=150, zorder=5, label='Centers')
    ax.set_title(title)
    ax.set_xlabel('tsne_1')
    ax.set_ylabel('tsne_2')
    ax.legend()

plt.suptitle('K-Means vs K-Medoid at K=6', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_k6.png', dpi=150)
plt.show()

In [ ]:
# Metric comparison at K=6
km6_idx = K_values.index(6)
print('=== Metric Comparison at K=6 ===')
print(f"{'Metric':<15} {'K-Means':>12} {'K-Medoid':>12}")
print('-' * 42)
for metric in ['Inertia', 'Purity', 'NMI', 'Silhouette']:
    km_val = kmeans_results[metric][km6_idx]
    kmed_val = kmedoid_results[metric][km6_idx]
    print(f"{metric:<15} {str(km_val):>12} {str(kmed_val):>12}")